# 복습


In [1]:
import torch

x = torch.tensor(2.0, requires_grad=True)   # "이 숫자의 기울기를 구할 거다" 표시
y = x ** 3

print("y      =", y.item())     # 8.0
print("grad 전:", x.grad)        # None ← 아직 안 구했으니 비어 있다

y.backward()                     # 기록을 되짚어 기울기 계산 → x.grad 에 담는다

print("grad 후:", x.grad)        # tensor(12.)

y      = 8.0
grad 전: None
grad 후: tensor(12.)


# step1

In [1]:
import torch
torch.manual_seed(0)

x = torch.linspace(-3, 3, 100)            # 입력 100개
y_clean = 3.0 * x + 2.0                   # 찾아내야 할 진짜 규칙
y = y_clean + 0.5 * torch.randn(100)      # 관측값에는 잡음이 섞여 있다

print(x.shape, y.shape)
print("정답:  w = 3.0,  b = 2.0")
print(y[:5])

torch.Size([100]) torch.Size([100])
정답:  w = 3.0,  b = 2.0
tensor([-7.5629, -7.3944, -6.7617, -6.6715, -5.8484])


# s2

In [2]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

y_pred = w * x + b                        # 순전파
loss = ((y_pred - y) ** 2).mean()         # MSE

print("loss  :", loss.item())
print("shape :", loss.shape)
print("w.grad:", w.grad)                  # backward() 전이라 None

loss  : 31.878366470336914
shape : torch.Size([])
w.grad: None


# S3

In [3]:
# 0 아님?
loss.backward()

print("w.grad:", w.grad.item())
print("b.grad:", b.grad.item())

w.grad: -18.362274169921875
b.grad: -4.037140846252441


In [4]:
with torch.no_grad():                     # 확인용 계산이므로 그래프를 만들지 않는다
    manual_gw = (2 * (y_pred - y) * x).mean()
    manual_gb = (2 * (y_pred - y)).mean()

print("손계산 w:", manual_gw.item(), " autograd:", w.grad.item())
print("손계산 b:", manual_gb.item(), " autograd:", b.grad.item())

손계산 w: -18.362272262573242  autograd: -18.362274169921875
손계산 b: -4.037140846252441  autograd: -4.037140846252441


In [22]:
lr = 0.1

with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad
    w.grad.zero_()
    b.grad.zero_()

y_pred = w * x + b
loss2 = ((y_pred - y) ** 2).mean()

print(f"{loss.item():.4f}  →  {loss2.item():.4f}")
print(f"w {w.item():.4f}   b {b.item():.4f}")

31.8784  →  7.0137
w 1.8362   b 0.4037


In [36]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.1

for step in range(100):
    if w.grad is not None:                # 첫 스텝에는 grad 가 아직 없다
        w.grad.zero_()
        b.grad.zero_()

    y_pred = w * x + b
    loss = ((y_pred - y) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    if step % 20 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}  w {w.item():.3f}  b {b.item():.3f}")

print(f"\n최종  w {w.item():.4f}   b {b.item():.4f}   (정답 3.0 / 2.0)")

step   0  loss 31.8784  w 1.836  b 0.404
step  20  loss 0.2629  w 3.000  b 2.000
step  40  loss 0.2624  w 3.000  b 2.018
step  60  loss 0.2624  w 3.000  b 2.019
step  80  loss 0.2624  w 3.000  b 2.019

최종  w 2.9998   b 2.0186   (정답 3.0 / 2.0)


In [37]:
w.grad.zero_()
b.grad.zero_()

y_pred = w * x + b
loss = ((y_pred - y) ** 2).mean()
loss.backward()

with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad


print(f"\n최종  w {w.item():.4f}   b {b.item():.4f}   (정답 3.0 / 2.0)")


최종  w 2.9998   b 2.0186   (정답 3.0 / 2.0)


In [44]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.3

for step in range(40):
    if w.grad is not None:
        w.grad.zero_(); b.grad.zero_()

    loss = ((w * x + b - y) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    print(f"step {step:2d}  loss {loss.item():>11.3e}  w {w.item():>11.3e}")

step  0  loss   3.188e+01  w   5.509e+00
step  1  loss   2.018e+01  w   9.014e-01
step  2  loss   1.384e+01  w   4.755e+00
step  3  loss   9.706e+00  w   1.532e+00
step  4  loss   6.859e+00  w   4.227e+00
step  5  loss   4.875e+00  w   1.973e+00
step  6  loss   3.489e+00  w   3.859e+00
step  7  loss   2.519e+00  w   2.282e+00
step  8  loss   1.841e+00  w   3.600e+00
step  9  loss   1.367e+00  w   2.497e+00
step 10  loss   1.035e+00  w   3.420e+00
step 11  loss   8.027e-01  w   2.648e+00
step 12  loss   6.403e-01  w   3.294e+00
step 13  loss   5.268e-01  w   2.754e+00
step 14  loss   4.473e-01  w   3.205e+00
step 15  loss   3.917e-01  w   2.828e+00
step 16  loss   3.529e-01  w   3.144e+00
step 17  loss   3.257e-01  w   2.879e+00
step 18  loss   3.066e-01  w   3.100e+00
step 19  loss   2.933e-01  w   2.916e+00
step 20  loss   2.840e-01  w   3.070e+00
step 21  loss   2.775e-01  w   2.941e+00
step 22  loss   2.730e-01  w   3.049e+00
step 23  loss   2.698e-01  w   2.959e+00
step 24  loss   

In [53]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.1

for step in range(50):
    if w.grad is not None:
        w.grad.zero_(); b.grad.zero_()

    y_pred = w * x + b
    loss = ((y_pred - y) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    if step % 10 == 0:
        print(f"step {step:2d}  loss {loss.item():.4f}  w {w.item():.3f}")

step  0  loss 31.8784  w 1.836
step 10  loss 0.3093  w 3.000
step 20  loss 0.2629  w 3.000
step 30  loss 0.2624  w 3.000
step 40  loss 0.2624  w 3.000


In [54]:
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1

for step in range(3):
    loss = ((w * x - y) ** 2).mean()
    loss.backward()

    w.grad.zero_()